In [ ]:
# 라이브러리, 데이터
import streamlit as st
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import koreanize_matplotlib

df = pd.read_csv("C:/Users/SBA/Downloads/kobis_20072026.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24241 entries, 0 to 24240
Data columns (total 18 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   순번      24241 non-null  int64
 1   영화명     24241 non-null  str  
 2   감독      20223 non-null  str  
 3   제작사     8246 non-null   str  
 4   수입사     12870 non-null  str  
 5   배급사     24204 non-null  str  
 6   개봉일     24240 non-null  str  
 7   영화유형    24241 non-null  str  
 8   영화형태    24238 non-null  str  
 9   국적      24241 non-null  str  
 10  전국      24241 non-null  str  
 11  전국.1    24220 non-null  str  
 12  전국.2    24241 non-null  str  
 13  서울      24214 non-null  str  
 14  서울.1    24241 non-null  str  
 15  장르      24110 non-null  str  
 16  등급      24239 non-null  str  
 17  영화구분    24241 non-null  str  
dtypes: int64(1), str(17)
memory usage: 7.8 MB


In [ ]:
# 분석방향성과 다른 데이터 확인
df["장르"].value_counts()
print( sum( df["장르"].isna() ) )
print( sum( df["장르"].isnull() ) )
df[ df["장르"] != "성인물(에로)" ].info()


131
131
<class 'pandas.DataFrame'>
Index: 18379 entries, 0 to 24240
Data columns (total 18 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   순번      18379 non-null  int64
 1   영화명     18379 non-null  str  
 2   감독      16518 non-null  str  
 3   제작사     6498 non-null   str  
 4   수입사     10640 non-null  str  
 5   배급사     18353 non-null  str  
 6   개봉일     18378 non-null  str  
 7   영화유형    18379 non-null  str  
 8   영화형태    18379 non-null  str  
 9   국적      18379 non-null  str  
 10  전국      18379 non-null  str  
 11  전국.1    18358 non-null  str  
 12  전국.2    18379 non-null  str  
 13  서울      18352 non-null  str  
 14  서울.1    18379 non-null  str  
 15  장르      18248 non-null  str  
 16  등급      18377 non-null  str  
 17  영화구분    18379 non-null  str  
dtypes: int64(1), str(17)
memory usage: 6.1 MB


In [ ]:
df["영화형태"].value_counts() # 유의미하나 데이터 미비
df["영화유형"].value_counts() # 단일값
df["영화구분"].value_counts() # 단일값

영화구분
일반영화       18063
독립/예술영화     6178
Name: count, dtype: int64

In [ ]:
for col in df.columns[2:-1]:
    if type(col) == str:
        print(col)
        print( df[col].unique() )
        print("="*80)

# 순번, 감독, 제작사, 수입사, 배급사는 유의미하지 않을 것으로 판단.
# 개봉일 데이터타입은 단순 문자열(숫자 사이를 '-'으로 구분)
# 영화유형은 전부 개봉영화, 영화구분은 전부 일반영화
# 영화형태, 영화구분에 'nan'이 결측치가 아닌 입력값으로 존재
# 장르가 "성인물(에로)"인 데이터와 결측치인 데이터는 이상치로 판단.
# 10.전국: 전국 스크린수
# 11.전국.1: 전국 매출액
# 12.전국.2: 전국 관객수
# 13.서울: 서울 매출액
# 14.서울.1: 서울 관객수

감독
<ArrowStringArray>
[        '김한민',         '장항준',         '이병헌',         '김용화',         '윤제균',
 '안소니 루소,조 루소', '크리스 벅,제니퍼 리',     '제임스 카메론',         '류승완',         '김성수',
 ...
         '카즈야',          '도희',         '이주이',         '이지연',         '김소리',
   '그레고리 하타나카',   '알렉세이 시도로프',      '오카다 히로',         '허재형',       '레 탄 선']
Length: 8326, dtype: str
제작사
<ArrowStringArray>
[                           '(주)빅스톤픽쳐스',
                 '(주)온다웍스,(주)비에이엔터테인먼트',
       '(주)어바웃잇,영화사 해그림 주식회사,(주)씨제이이엔엠',
               '리얼라이즈픽쳐스(주),(주)덱스터스튜디오',
                  '(주)제이케이필름,(주)씨제이이엔엠',
                                    nan,
                        '이십세기폭스필름코퍼레이션',
                      '(주)외유내강,(주)필름케이',
                          '(주)하이브미디어코프',
                             '(주)케이퍼필름',
 ...
                            '(주)데이드림서울',
                     '(주)룬컴퍼니,영화사 산들바람',
                              '(주)플라이어',
                 '영화사 레드카펫,주식회사 루믹스미디어',
                           '(주)영화사단막극장',
 

In [ ]:
# 데이터 추출 가능성 확인
df_ticket = df.copy()
df_ticket = df_ticket[df_ticket["장르"] != "성인물(에로)"]

target_columns = ["전국", "전국.1", "전국.2", "서울", "서울.1"]

# 지정한 모든 열의 콤마를 제거하고 int형으로 변환
df_ticket = df_ticket.dropna(axis=0, subset=["전국.1", "서울"] )
for col in target_columns:
    df_ticket[col] = df_ticket[col].str.replace(",", "", regex=True).astype(int)

df_ticket = df_ticket.drop( df_ticket[ df_ticket["전국.1"] == 0 ].index )
df_ticket = df_ticket.drop( df_ticket[ df_ticket["서울"] == 0 ].index )

df_ticket.info()

# 부가가치세 10%, 영화발전기금 3%, 남은 87%를 극장과 배급사가 50:50
# "전국.1" / 전국.2 /0.83 / 0.5 = 해당영화의 해당연도 평균 티켓값
df_ticket["예측티켓값"] = df_ticket["전국.1"] / df_ticket["전국.2"] / 0.83 / 0.5

<class 'pandas.DataFrame'>
Index: 12521 entries, 0 to 24223
Data columns (total 18 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   순번      12521 non-null  int64
 1   영화명     12521 non-null  str  
 2   감독      11645 non-null  str  
 3   제작사     4609 non-null   str  
 4   수입사     7705 non-null   str  
 5   배급사     12501 non-null  str  
 6   개봉일     12521 non-null  str  
 7   영화유형    12521 non-null  str  
 8   영화형태    12521 non-null  str  
 9   국적      12521 non-null  str  
 10  전국      12521 non-null  int64
 11  전국.1    12521 non-null  int64
 12  전국.2    12521 non-null  int64
 13  서울      12521 non-null  int64
 14  서울.1    12521 non-null  int64
 15  장르      12471 non-null  str  
 16  등급      12521 non-null  str  
 17  영화구분    12521 non-null  str  
dtypes: int64(6), str(12)
memory usage: 3.9 MB


In [ ]:
df_ticket["예측티켓값"].describe()

count     12521.000000
mean      18202.110427
std        8655.705929
min          16.046130
25%       14491.911871
50%       17966.232767
75%       20248.821372
max      168674.698795
Name: 예측티켓값, dtype: float64

In [ ]:
df_ticket.sort_values("예측티켓값")

,순번,영화명,감독,제작사,수입사,배급사,개봉일,영화유형,영화형태,국적,전국,전국.1,전국.2,서울,서울.1,장르,등급,영화구분,예측티켓값
1686,1687,인플루언스,이재규,"리얼라이즈픽쳐스(주),윈저엔터테인먼트",NaN,NaN,2010-08-19,개봉영화,장편,한국,3,1529006,229610,1310000,175,드라마,15세이상관람가,일반영화,16.046130
1679,1680,원스,존 카니,NaN,"(주)제이앤씨미디어그룹,(주)영화사 진진","(주)제이앤씨미디어그룹,와이드 릴리즈(주),(주)영화사 진진",2007-09-20,개봉영화,장편,아일랜드,10,45419094,232459,44139500,180826,드라마,전체관람가,독립/예술영화,470.808186
4816,4817,추적자,무로가 아츠시,NaN,스크린조이,스크린조이,2014-03-06,개봉영화,장편,일본,5,6310000,10502,520000,104,범죄,청소년관람불가,일반영화,1447.802255
5300,5301,워리어스 레인보우: 항전의 시작,위덕성,NaN,스크린조이,스크린조이,2014-02-20,개봉영화,장편,대만,5,5070000,7640,950000,190,드라마,청소년관람불가,일반영화,1599.066423
6166,6167,거짓말 섹스가 좋아2,진달래,(주)케이알씨지,NaN,(주)케이알씨지,2013-12-24,개봉영화,장편,한국,5,2980000,4490,500000,100,멜로/로맨스,청소년관람불가,일반영화,1599.270132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3470,3471,엔하이픈 브이알콘서트 : 이머전,김경국,(주)어메이즈브이알코리아,NaN,(주)어메이즈브이알코리아,2025-08-08,개봉영화,장편,한국,1,945066000,29022,945066000,29022,드라마,전체관람가,일반영화,78466.937836
10328,10329,밀라노 두오모 콘서트,리카르도 샤이,NaN,일미디어,일미디어,2022-01-07,개봉영화,장편,이탈리아,6,6469200,190,4699700,137,공연,전체관람가,일반영화,82044.388079
6263,6264,앤팀 브이알 콘서트 : 바운드리스,김경국,(주)어메이즈브이알코리아,NaN,(주)어메이즈브이알코리아,2026-06-26,개봉영화,장편,한국,1,151404000,4207,151404000,4207,드라마,전체관람가,일반영화,86719.495047
7239,7240,요루시카 '달과 고양이의 댄스' 2024 라이브,NaN,NaN,NaN,씨제이 씨지브이(CJ CGV)(주),2024-11-29,개봉영화,장편,일본,3,108339000,2211,89376000,1824,공연,전체관람가,일반영화,118072.289157


In [ ]:
# 날짜 -> 타임스탬프 변환
df_date = pd.DataFrame({"raw_date": df_ticket["개봉일"] })
df_date["clean_date"] = pd.to_datetime(df_date["raw_date"], errors='coerce')

print(df_date)

df_ticket["개봉연도"] = df_date["clean_date"].dt.year.tolist()
df_ticket["개봉월"] = df_date["clean_date"].dt.month.tolist()

df_ticket

         raw_date clean_date
0      2014-07-30 2014-07-30
1      2026-02-04 2026-02-04
2      2019-01-23 2019-01-23
3      2017-12-20 2017-12-20
4      2014-12-17 2014-12-17
...           ...        ...
24106  2021-03-27 2021-03-27
24109  2015-08-05 2015-08-05
24195  2014-04-24 2014-04-24
24203  2010-06-10 2010-06-10
24223  2021-03-26 2021-03-26

[12521 rows x 2 columns]


,순번,영화명,감독,제작사,수입사,배급사,개봉일,영화유형,영화형태,국적,...,전국.1,전국.2,서울,서울.1,장르,등급,영화구분,예측티켓값,개봉연도,개봉월
0,1,명량,김한민,(주)빅스톤픽쳐스,NaN,(주)씨제이이엔엠,2014-07-30,개봉영화,장편,한국,...,135748398910,17613682,33121225810,4163666,사극,15세이상관람가,일반영화,18571.050374,2014,7
1,2,왕과 사는 남자,장항준,"(주)온다웍스,(주)비에이엔터테인먼트",NaN,(주)쇼박스,2026-02-04,개봉영화,장편,한국,...,162898938670,16880990,34840190620,3494144,사극,12세이상관람가,일반영화,23252.638800,2026,2
2,3,극한직업,이병헌,"(주)어바웃잇,영화사 해그림 주식회사,(주)씨제이이엔엠",NaN,(주)씨제이이엔엠,2019-01-23,개봉영화,장편,한국,...,139647979516,16264944,31858660536,3638287,코미디,15세이상관람가,일반영화,20688.737413,2019,1
3,4,신과함께-죄와 벌,김용화,"리얼라이즈픽쳐스(주),(주)덱스터스튜디오",NaN,롯데쇼핑㈜롯데엔터테인먼트,2017-12-20,개봉영화,장편,한국,...,115698654137,14410754,27530825087,3346172,판타지,12세이상관람가,일반영화,19346.103450,2017,12
4,5,국제시장,윤제균,"(주)제이케이필름,(주)씨제이이엔엠",NaN,(주)씨제이이엔엠,2014-12-17,개봉영화,장편,한국,...,110828014630,14245998,25842519330,3233946,드라마,12세이상관람가,일반영화,18745.998486,2014,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24106,24107,2021 첫 섹스,김동해,라이어 시네마,NaN,케이엘 픽쳐스,2021-03-27,개봉영화,장편,한국,...,2000,1,2000,1,멜로/로맨스,청소년관람불가,일반영화,4819.277108,2021,3
24109,24110,2047: 지구 최후의 날,알렉산드로 카폰,NaN,(주)소나무픽쳐스,(주)소나무픽쳐스,2015-08-05,개봉영화,장편,이탈리아,...,8000,1,8000,1,액션,15세이상관람가,일반영화,19277.108434,2015,8
24195,24196,F.O.E.: 에프.오.이.,키오니 왁스먼,NaN,라인트리엔터테인먼트,라인트리엔터테인먼트,2014-04-24,개봉영화,장편,미국,...,2000,1,2000,1,액션,청소년관람불가,일반영화,4819.277108,2014,4
24203,24204,H2: 어느 살인마의 가족이야기,롭 좀비,"디멘션 필름즈,스펙타클 엔터테인먼트 그룹,트랜카스 인태내셔널 필름",(주)누리픽쳐스,(주)누리픽쳐스,2010-06-10,개봉영화,장편,미국,...,8000,1,8000,1,공포(호러),청소년관람불가,일반영화,19277.108434,2010,6


In [ ]:
for year in range(2017,2027):
    print( df_ticket["예측티켓값"][df_ticket["개봉연도"] == year].describe() )

count      936.000000
mean     15884.000328
std       7073.929332
min       4535.790220
25%      12170.721345
50%      17198.768066
75%      18739.149671
max      59394.582491
Name: 예측티켓값, dtype: float64
count     1006.000000
mean     14852.096219
std       7953.995440
min       2409.638554
25%      11503.108576
50%      17157.380688
75%      19457.463378
max      59082.319845
Name: 예측티켓값, dtype: float64
count      838.000000
mean     16581.674032
std       8776.226812
min       2409.638554
25%      12290.928420
50%      18200.569722
75%      19861.766891
max      72289.156627
Name: 예측티켓값, dtype: float64
count      815.000000
mean     17925.661019
std      10194.623077
min       2409.638554
25%      12048.192771
50%      18818.536409
75%      20507.383924
max      73936.416988
Name: 예측티켓값, dtype: float64
count      937.000000
mean     17504.656143
std       8578.849012
min       2409.638554
25%      12048.192771
50%      18263.835917
75%      21418.014400
max      71834.002677
Name: 예측

In [ ]:
# 데이터 정제

# 컬럼 제한
target_columns = [ "영화명", "영화형태", "국적", "전국", "전국.1", "전국.2", "서울", "서울.1", "장르", "등급", "예측티켓값", "개봉연도", "개봉월" ]
df_ticket = df_ticket[target_columns]

# "장르"가 "성인물(에로)", "공연", "뮤지컬"인 데이터와 결측치인 데이터는 이상치로 판단, 삭제.
df_ticket = df_ticket[ ~df_ticket["장르"].str.contains("성인물(에로)|공연|뮤지컬", na=False ) ]
df_ticket = df_ticket.dropna(subset=["장르"])

# "예측티켓값"이 10000 미만인 데이터는 이상치로 판단, 삭제.
df_ticket = df_ticket.drop( df_ticket[ df_ticket["예측티켓값"] < 10000 ].index )

# "전국"이 1이하고, "전국.2"가 100 미만인 데이터, 또는 "전국"이 10 이하고, "전국.2"가 50 이하인 데이터는 이상치로 판단, 삭제.
df_ticket = df_ticket.drop( df_ticket[ df_ticket["전국"] <= 10 ][ df_ticket["전국.2"] <= 100 ].index )

# 콘서트, 뮤지컬 제외
df_ticket = df_ticket[ ~df_ticket["영화명"].str.contains("콘서트|뮤지컬", na=False ) ]

# "등급"이 "청소년관람불가"이면서 "장르"가 "멜로/로맨스", "드라마", "기타", "코미디"인 데이터 중 일부를 이상치로 판단, 삭제.
df_ticket = df_ticket.drop( df_ticket[ df_ticket["등급"] == "청소년관람불가" ][ df_ticket["장르"] == "멜로/로맨스" ].index )
df_ticket = df_ticket.drop( df_ticket[ df_ticket["등급"] == "청소년관람불가" ][ df_ticket["장르"] == "드라마" ][ df_ticket["전국"] < 30 ].index )
df_ticket = df_ticket.drop( df_ticket[ df_ticket["등급"] == "청소년관람불가" ][ df_ticket["장르"] == "기타" ][ df_ticket["전국"] < 30 ].index )
df_ticket = df_ticket.drop( df_ticket[ df_ticket["등급"] == "청소년관람불가" ][ df_ticket["장르"] == "코미디" ][ df_ticket["예측티켓값"] < 17000 ].index )

df_ticket

C:\Users\SBA\AppData\Local\Temp\ipykernel_5524\1696977932.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_ticket = df_ticket[ ~df_ticket["장르"].str.contains("성인물(에로)|공연|뮤지컬", na=False ) ]
C:\Users\SBA\AppData\Local\Temp\ipykernel_5524\1696977932.py:15: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_ticket = df_ticket.drop( df_ticket[ df_ticket["전국"] <= 1 ][ df_ticket["전국.2"] < 100 ].index )
C:\Users\SBA\AppData\Local\Temp\ipykernel_5524\1696977932.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_ticket = df_ticket.drop( df_ticket[ df_ticket["전국"] <= 10 ][ df_ticket["전국.2"] <= 50 ].index )
C:\Users\SBA\AppData\Local\Temp\ipykernel_5524\1696977932.py:22: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_ticket = df_ticket.drop( df_ticket[ df_ticket["등급"] == "청소년관람불가" ][ df_ticket["장르"] == "멜로

,영화명,영화형태,국적,전국,전국.1,전국.2,서울,서울.1,장르,등급,예측티켓값,개봉연도,개봉월
0,명량,장편,한국,1587,135748398910,17613682,33121225810,4163666,사극,15세이상관람가,18571.050374,2014,7
1,왕과 사는 남자,장편,한국,1724,162898938670,16880990,34840190620,3494144,사극,12세이상관람가,23252.638800,2026,2
2,극한직업,장편,한국,1978,139647979516,16264944,31858660536,3638287,코미디,15세이상관람가,20688.737413,2019,1
3,신과함께-죄와 벌,장편,한국,1912,115698654137,14410754,27530825087,3346172,판타지,12세이상관람가,19346.103450,2017,12
4,국제시장,장편,한국,966,110828014630,14245998,25842519330,3233946,드라마,12세이상관람가,18745.998486,2014,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12382,고골: 악령과의 전쟁,장편,러시아,5,220000,51,84000,28,미스터리,15세이상관람가,10394.519253,2019,10
12675,초능력 에볼루션,장편,미국,19,240900,36,126400,14,SF,15세이상관람가,16124.497992,2021,5
12775,레저렉션 이펙트,장편,기타,21,189940,30,100940,12,SF,청소년관람불가,15256.224900,2021,6
12882,스페이스 시그널,장편,미국,18,118480,24,53540,8,액션,12세이상관람가,11895.582329,2021,5


In [ ]:
df_ticket["예측티켓값"].describe()

count     8787.000000
mean     18942.389962
std       3994.180188
min      10020.570085
25%      17042.773375
50%      18594.966926
75%      20645.523873
max      75849.024674
Name: 예측티켓값, dtype: float64

In [ ]:
# df_ticket[ df_ticket["장르"].str.contains("다큐멘터리", na=False ) ]
# df_ticket[ df_ticket["등급"] == "청소년관람불가" ][ df_ticket["장르"] == "기타" ].sort_values("전국")
df_ticket.sort_values("예측티켓값", ascending=False).tail(60)

,영화명,영화형태,국적,전국,전국.1,전국.2,서울,서울.1,장르,등급,예측티켓값,개봉연도,개봉월
9242,크리미널 트위스트,장편,영국,50,2444000,524,650000,130,액션,15세이상관람가,11238.848524,2022,6
5343,죽이는 여자,장편,미국,1,34371000,7395,1491000,1395,공포(호러),청소년관람불가,11199.687187,2020,8
10753,환생령,장편,말레이시아,5,645000,139,16000,8,공포(호러),12세이상관람가,11181.416313,2018,9
9405,더 워닝,장편,스페인,32,2068500,448,720500,140,스릴러,15세이상관람가,11125.753012,2020,5
9955,애플시드:엑스머시나,장편,일본,1,1204000,261,1204000,261,SF,12세이상관람가,11115.727277,2008,6
11287,불륜관계,장편,중국,6,392000,85,116000,29,멜로/로맨스,15세이상관람가,11112.686038,2016,7
9229,몬스터 아일랜드,장편,미국,50,2452000,532,600000,120,SF,15세이상관람가,11106.078449,2022,4
10808,탐정 당인: 차이나타운 살인사건,장편,중국,1,613000,133,613000,133,액션,15세이상관람가,11106.078449,2018,3
4952,영웅: 천하의 시작,장편,중국,54,44066600,9575,16875000,2228,액션,15세이상관람가,11089.773192,2014,3
9225,데스위시 더 게임,장편,미국,50,2453000,533,600000,120,액션,15세이상관람가,11089.762427,2022,4


In [ ]:
df_ticket[ df_ticket["등급"] == "청소년관람불가" ][ df_ticket["장르"] == "코미디" ][ df_ticket["예측티켓값"] < 17000 ]

C:\Users\SBA\AppData\Local\Temp\ipykernel_5524\2920374540.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_ticket[ df_ticket["등급"] == "청소년관람불가" ][ df_ticket["장르"] == "코미디" ][ df_ticket["예측티켓값"] < 17000 ]
C:\Users\SBA\AppData\Local\Temp\ipykernel_5524\2920374540.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_ticket[ df_ticket["등급"] == "청소년관람불가" ][ df_ticket["장르"] == "코미디" ][ df_ticket["예측티켓값"] < 17000 ]


,영화명,영화형태,국적,전국,전국.1,전국.2,서울,서울.1,장르,등급,예측티켓값,개봉연도,개봉월
